# 06 — E-Iso-1..6: Fault Injection & Attack Containment

RFC-008 §D8. Six attack scenarios in position 2 of a 3-node linear pipeline.
Each attack traps on every `process()` call; the isolation invariant is:
source throughput unaffected, attack contained, no cross-node contamination.

**Inputs**: `eval/results/e-iso-{1..6}/shakedown-macos-<ts>/` produced by
`eval/scripts/run-e-iso-shakedown.sh --all`.

In [ ]:
import json, os, glob, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'eval').is_dir():
    REPO = REPO.parent

ATTACKS = [
    ('buffer-overflow', 1),
    ('cross-read', 2),
    ('fs-access', 3),
    ('infinite-loop', 4),
    ('memory-exhaust', 5),
    ('panic', 6),
]

def find_shakedown(iso_num):
    """Find the newest shakedown directory for a given E-Iso experiment."""
    pattern = str(REPO / f'eval/results/e-iso-{iso_num}/shakedown-macos-*')
    candidates = sorted(glob.glob(pattern))
    if not candidates:
        return None
    return Path(candidates[-1])

# Load all shakedown results
results = []
for attack_name, iso_num in ATTACKS:
    sd = find_shakedown(iso_num)
    if sd is None:
        print(f'MISSING: e-iso-{iso_num} ({attack_name})')
        continue
    sj = sd / 'shakedown.json'
    if not sj.exists():
        print(f'MISSING shakedown.json: {sj}')
        continue
    data = json.loads(sj.read_text())
    data['attack_name'] = attack_name
    data['iso_num'] = iso_num
    data['shakedown_dir'] = str(sd)
    results.append(data)

df = pd.DataFrame(results)
print(f'Loaded {len(df)} attack results')
df[['attack_name', 'contained', 'source_throughput_pct', 'attacker_state', 'other_nodes_healthy']]

## Per-node throughput timeline

For each attack, parse `stdout.log` to extract timestamps of ERROR events
(attack node traps) and show the source emitting at constant rate while the
attack node traps on every message.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
axes_flat = axes.flatten()

ANSI_RE = re.compile(r'\x1b\[[0-9;]*m')
TS_RE = re.compile(r'^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z)')

for idx, (attack_name, iso_num) in enumerate(ATTACKS):
    ax = axes_flat[idx]
    sd = find_shakedown(iso_num)
    if sd is None:
        ax.set_title(f'E-Iso-{iso_num}: MISSING')
        continue

    log_path = sd / 'run-1' / 'stdout.log'
    if not log_path.exists():
        ax.set_title(f'E-Iso-{iso_num}: no log')
        continue

    # Parse timestamps from log
    trap_times = []
    pipeline_start = None
    pipeline_end = None

    for line in log_path.read_text().splitlines():
        clean = ANSI_RE.sub('', line)
        m = TS_RE.match(clean)
        if not m:
            continue
        ts_str = m.group(1)
        if 'Pipeline running' in clean:
            pipeline_start = pd.Timestamp(ts_str)
        if 'Pipeline completed' in clean:
            pipeline_end = pd.Timestamp(ts_str)
        if 'unrecoverable error' in clean and 'node=' in clean:
            trap_times.append(pd.Timestamp(ts_str))

    if not pipeline_start or not trap_times:
        ax.set_title(f'E-Iso-{iso_num}: no data')
        continue

    # Convert to relative seconds
    trap_secs = np.array([(t - pipeline_start).total_seconds() for t in trap_times])
    end_sec = (pipeline_end - pipeline_start).total_seconds() if pipeline_end else trap_secs[-1] + 0.1

    # Source emits at constant rate: 100 msg/s for 2s
    total_msgs = 200
    source_rate = 100.0
    source_times = np.array([i / source_rate for i in range(total_msgs)])

    # Bin into 100ms windows for throughput
    bin_width = 0.1
    max_t = max(end_sec, source_times[-1]) + bin_width
    bins = np.arange(0, max_t + bin_width, bin_width)

    # Source throughput per bin
    src_counts, _ = np.histogram(source_times, bins=bins)
    src_thr = src_counts / bin_width

    # Attack trap throughput per bin
    trap_counts, _ = np.histogram(trap_secs, bins=bins)
    trap_thr = trap_counts / bin_width

    bin_centers = (bins[:-1] + bins[1:]) / 2

    ax.step(bin_centers, src_thr, where='mid', label='Source (emitted)', color='tab:blue', linewidth=1.5)
    ax.step(bin_centers, trap_thr, where='mid', label='Attack (traps)', color='tab:red', linewidth=1.5, linestyle='--')
    ax.axhline(y=0, color='tab:green', linewidth=1, linestyle=':', label='Sink (received)', alpha=0.7)

    ax.set_title(f'E-Iso-{iso_num}: {attack_name}', fontsize=10)
    ax.set_xlabel('Time (s)')
    if idx % 3 == 0:
        ax.set_ylabel('Throughput (msg/s)')
    ax.set_ylim(-10, 150)
    ax.grid(True, alpha=0.3)

# Shared legend
handles = [
    mpatches.Patch(color='tab:blue', label='Source (emitted)'),
    mpatches.Patch(color='tab:red', label='Attack (trapped)'),
    mpatches.Patch(color='tab:green', label='Sink (received = 0)'),
]
fig.legend(handles=handles, loc='upper center', ncol=3, fontsize=10, frameon=False)
fig.suptitle('Per-Node Throughput: Source Unaffected, Attack Contained', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## Summary table (readiness matrix)

Matches the format in `docs/status/canonical-readiness.md`.

In [ ]:
summary = df[['attack_name', 'iso_num', 'contained', 'source_throughput_pct',
              'attacker_state', 'other_nodes_healthy', 'error_count', 'total_messages']].copy()
summary['status'] = summary.apply(
    lambda r: '\U0001f7e2' if r['contained'] and r['other_nodes_healthy'] else '\U0001f534', axis=1
)
summary = summary.rename(columns={
    'attack_name': 'Attack',
    'iso_num': 'E-Iso-N',
    'contained': 'Contained',
    'source_throughput_pct': 'Source Thr %',
    'attacker_state': 'Attacker State',
    'other_nodes_healthy': 'Healthy Nodes',
    'error_count': 'Traps',
    'total_messages': 'Total Msgs',
    'status': 'Status',
})
summary[['Status', 'Attack', 'E-Iso-N', 'Contained', 'Source Thr %', 'Attacker State', 'Traps', 'Healthy Nodes']]

In [ ]:
# Verify all attacks are contained
assert df['contained'].all(), 'Not all attacks were contained!'
assert df['other_nodes_healthy'].all(), 'Some non-attack nodes reported errors!'
assert (df['source_throughput_pct'] >= 99.0).all(), 'Source throughput dropped below 99%!'
print('All 6 attacks CONTAINED. Source throughput >= 99% for all. No cross-node contamination.')